# DegreeDetailExtract — v5 Training Notebook

**What's new in v5 (vs v4):**
- 60 real labeled certificates mixed directly into training data
- Written-out date formats (`on the 15th day of the month June, two thousand and nineteen`)
- Authority name: 60% empty / 20% title-only / 20% with person name (matches real cert distribution)
- Specialization auto-empty for BCA, MBBS, BBA, MBA, LLB, BCA, B.Ed., Ph.D etc.
- 6 new prose templates: `has_been_awarded`, `of_the_university`, `written_date_signed`,
  `title_only_authority`, `no_specialization`, `label_grid_real_style`
- 30 prose templates total (was 24)
- Expanded PASS_CLASSES: added A+, A, O, Pass Division, Honours Class I/II
- Fixed checkpoint saving: never rmtree epoch folders (Drive FUSE async race bug)

**How to run:**
1. Run Section 0 (environment), Section 1 (clone + fonts)
2. Run Section 2 (generate dataset — 5,500 synthetic + 60 real = ~5,560 images)
3. Run Section 3 cells in order (load model, build dataset, train)
4. If Colab disconnects: run Section 4 (resume) — it is fully self-contained
5. Run Section 5 (evaluation), Section 6 (real cert inference), Section 8 (real-world eval)


## Section 0 — Environment Setup

In [ ]:
# System fonts for certificate rendering
!apt-get install -y --fix-missing fonts-dejavu fonts-liberation fonts-noto ttf-mscorefonts-installer -q

# Python packages
!pip install -q --prefer-binary tokenizers
!pip install -q --prefer-binary transformers datasets accelerate sentencepiece editdistance \\
    Pillow faker reportlab

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DegreeDetailExtract', exist_ok=True)
print('Google Drive mounted.')

## Section 1 — Clone Repo & Download Fonts

In [ ]:
import os, sys

REPO = '/content/DegreeDetailExtract'
if not os.path.exists(REPO):
    !git clone https://github.com/Vrushti33/DegreeDetailExtract.git {REPO}
else:
    !git -C {REPO} pull

sys.path.insert(0, REPO)
print('Repo ready:', REPO)

In [ ]:
# Download calligraphy/formal certificate fonts from Google Fonts (OFL)
import os, urllib.request

FONT_DIR = '/content/DegreeDetailExtract/cert_fonts'
os.makedirs(FONT_DIR, exist_ok=True)

FONTS = {
    # Serif / formal
    'IMFellEnglish-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/imfellenglish/IMFellEnglish-Regular.ttf',
    'Cinzel-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/cinzel/static/Cinzel-Regular.ttf',
    'Playfair_Display_Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/playfairdisplay/static/PlayfairDisplay-Regular.ttf',
    'EBGaramond-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/ebgaramond/static/EBGaramond-Regular.ttf',
    'UnifrakturMaguntia.ttf':
        'https://github.com/google/fonts/raw/main/ofl/unifrakturmaguntia/UnifrakturMaguntia.ttf',
    # Script / cursive
    'GreatVibes-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/greatvibes/GreatVibes-Regular.ttf',
    'Parisienne-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/parisienne/Parisienne-Regular.ttf',
    'Allura-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/allura/Allura-Regular.ttf',
    'PinyonScript-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/pinyonscript/PinyonScript-Regular.ttf',
    'AlexBrush-Regular.ttf':
        'https://github.com/google/fonts/raw/main/ofl/alexbrush/AlexBrush-Regular.ttf',
}

for fname, url in FONTS.items():
    dst = f'{FONT_DIR}/{fname}'
    if not os.path.exists(dst):
        try:
            urllib.request.urlretrieve(url, dst)
            print(f'  Downloaded: {fname}')
        except Exception as e:
            print(f'  WARN: {fname} — {e}')
    else:
        print(f'  OK: {fname}')

print(f'Fonts in {FONT_DIR}:', len(os.listdir(FONT_DIR)))

## Section 2 — Generate v5 Dataset

**Expected time: 20-30 min** on T4 GPU Colab.

v5 dataset composition:
- **5,500 synthetic certificates** with improved prose + date formats
- **60 real labeled certificates** from `real_certs/` — included directly in training

This means the model trains on actual real certificate images with correct labels,
drastically reducing the synthetic-to-real domain gap without needing domain adaptation.

> Run the Force-Regenerate cell below ONLY if you need a fresh dataset.
> Otherwise, the main generation cell will restore from Drive if archive exists.


In [ ]:
# Force v5 dataset regeneration (delete existing archive)
# Run this ONLY if you need to regenerate from scratch
import os, shutil

DATASET  = '/content/dataset_v5'
ZIP_PATH = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v5_archive.zip'

if os.path.exists(DATASET):
    shutil.rmtree(DATASET)
    print(f'Deleted: {DATASET}')
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
    print(f'Deleted: {ZIP_PATH}')
print('Ready to regenerate.')

In [ ]:
# Section 2: Generate v5 Dataset OR restore from Drive archive
# v5: 5,500 synthetic certs + ALL 60 real labeled certs mixed into training
import os, sys, json, shutil, random, tempfile, subprocess

REPO    = '/content/DegreeDetailExtract'
DATASET = '/content/dataset_v5'
DRIVE_BASE  = '/content/drive/MyDrive/DegreeDetailExtract'
ZIP_PATH    = f'{DRIVE_BASE}/dataset_v5_archive.zip'
REAL_DIR    = f'{REPO}/real_certs'
REAL_META   = f'{REAL_DIR}/metadata.jsonl'

MIN_TRAIN_RECORDS = 4200   # 98% of expected ~4,300 synthetic train records

meta_train = f'{DATASET}/metadata_train.jsonl'
need_generate = (
    not os.path.exists(meta_train)
    or sum(1 for _ in open(meta_train)) < MIN_TRAIN_RECORDS
)

if not need_generate:
    count = sum(1 for _ in open(meta_train))
    print(f'Dataset already present ({count:,} train records). Skipping generation.')
else:
    # Try restoring from Drive archive first
    if os.path.exists(ZIP_PATH):
        print(f'Restoring dataset from {ZIP_PATH} ...')
        os.makedirs(DATASET, exist_ok=True)
        TMP = tempfile.mkdtemp(prefix='cert_extract_', dir='/content')
        try:
            subprocess.run(['unzip', '-qo', ZIP_PATH, '-d', TMP], check=True)
            for root, dirs, files in os.walk(TMP):
                dirs[:] = [d for d in dirs if not d.startswith('drive')]
                if 'metadata_train.jsonl' in files:
                    for item in os.listdir(root):
                        dst = os.path.join(DATASET, item)
                        if os.path.exists(dst):
                            if os.path.isdir(dst): shutil.rmtree(dst)
                            else: os.remove(dst)
                        shutil.move(os.path.join(root, item), DATASET)
                    break
            count = sum(1 for _ in open(meta_train)) if os.path.exists(meta_train) else 0
            if count >= MIN_TRAIN_RECORDS:
                print(f'Restored from archive: {count:,} train records.')
                need_generate = False
            else:
                print(f'Archive had only {count:,} records (need {MIN_TRAIN_RECORDS:,}). Regenerating...')
        finally:
            if os.path.exists(TMP): shutil.rmtree(TMP, ignore_errors=True)

if need_generate:
    print('Generating v5 synthetic dataset (5,500 certificates) ...')
    sys.path.insert(0, REPO)
    os.makedirs(f'{DATASET}/images', exist_ok=True)

    from generator.faker_fields import generate_fields
    from generator.fonts import download_certificate_fonts
    from generator.renderer_v3 import render_certificate_v3, init_bg_pool
    from generator.augment import augment_image

    # Download certificate fonts to /content/cert_fonts
    download_certificate_fonts('/content/cert_fonts')

    # Load real cert images as background textures
    n_bg = init_bg_pool(f'{REPO}/real_certs') if os.path.exists(f'{REPO}/real_certs') else 0
    print(f'  Loaded {n_bg} real backgrounds for texture')

    N = 5500
    records = []
    failed  = 0
    for i in range(N):
        fields = generate_fields()
        try:
            img = render_certificate_v3(fields, augment=True)
        except Exception as e:
            failed += 1
            continue
        fname = f'images/cert_{i+1:05d}.jpg'
        img.save(f'{DATASET}/{fname}', 'JPEG', quality=92)
        fields['file_name'] = fname
        records.append(fields)
        if (i + 1) % 500 == 0:
            print(f'  {i+1:,}/{N}  (skipped: {failed})', end='\r', flush=True)

    print(f'\n  Synthetic generation done: {len(records):,} images ({failed} skipped)')

    # ── Mix in the 60 real labeled certificates ──────────────────────────────
    real_records = []
    if os.path.exists(REAL_META):
        with open(REAL_META) as fp:
            for line in fp:
                rec = json.loads(line.strip())
                if not rec.get('file_name'):
                    continue
                src_img = f'{REAL_DIR}/{rec["file_name"]}'
                if not os.path.exists(src_img):
                    continue
                # Copy real image into dataset images folder
                ext = os.path.splitext(rec['file_name'])[-1].lower()
                new_fname = f'images/real_{len(real_records)+1:03d}{ext}'
                shutil.copy2(src_img, f'{DATASET}/{new_fname}')
                rec['file_name'] = new_fname
                rec['is_real'] = True
                real_records.append(rec)
        print(f'  Included {len(real_records)} real labeled certificates')

    all_records = records + real_records
    random.seed(42)
    random.shuffle(all_records)

    # Split: 87% train / 10% val / 3% test
    # Real certs are spread across splits proportionally
    n_train = int(len(all_records) * 0.87)
    n_val   = int(len(all_records) * 0.10)
    splits = {
        'train': all_records[:n_train],
        'val':   all_records[n_train:n_train + n_val],
        'test':  all_records[n_train + n_val:],
    }
    for split, recs in splits.items():
        with open(f'{DATASET}/metadata_{split}.jsonl', 'w') as fp:
            for r in recs:
                fp.write(json.dumps(r, ensure_ascii=False) + '\n')
        real_count = sum(1 for r in recs if r.get('is_real'))
        print(f'  {split:5s}: {len(recs):,} records ({real_count} real)')

print('\nDataset ready.')

# ── Save archive to Drive ──────────────────────────────────────────────────────
save_to_drive = True   # Set to False to skip (if you already have the archive)
if save_to_drive and not os.path.exists(ZIP_PATH):
    print('Saving dataset archive to Drive ...')
    tmp_zip = f'/content/dataset_v5_archive'
    shutil.make_archive(tmp_zip, 'zip', '/content', 'dataset_v5')
    shutil.move(f'{tmp_zip}.zip', ZIP_PATH)
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'Saved: {ZIP_PATH} ({size_mb:.0f} MB)')
elif os.path.exists(ZIP_PATH):
    print(f'Drive archive already exists: {ZIP_PATH}')

In [ ]:
# Preview 6 random certificates (3 synthetic, up to 3 real)
import random, json
from PIL import Image
import matplotlib.pyplot as plt

DATASET = '/content/dataset_v5'
with open(f'{DATASET}/metadata_train.jsonl') as fp:
    rows = [json.loads(l) for l in fp]

real_rows  = [r for r in rows if r.get('is_real')]
synth_rows = [r for r in rows if not r.get('is_real')]

samples = random.sample(synth_rows, min(3, len(synth_rows))) + random.sample(real_rows, min(3, len(real_rows)))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, row in zip(axes.flat, samples):
    img = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    ax.imshow(img)
    label = 'REAL' if row.get('is_real') else 'synthetic'
    ax.set_title(f"[{label}] {row.get('student_name','')[:25]}\n{row.get('course_name','')[:30]}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Total: {len(rows):,} train | {len(real_rows)} real | {len(synth_rows):,} synthetic')

## Section 3 — Fine-tune Donut (10 Epochs, v5)

> **Expected time**: ~5.5 hrs on T4 GPU for 10 epochs.
> Each epoch ~30-35 min. Checkpoints saved every epoch to Google Drive.
> If Colab disconnects, use Section 4 to resume.


In [ ]:
# Section 3 — Load Model & Processor (v5)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, gc
from transformers import DonutProcessor, VisionEncoderDecoderModel

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = 'naver-clova-ix/donut-base'
DATASET    = '/content/dataset_v5'

# v5 field list (same as v4, unchanged schema)
FIELDS = ['student_name', 'university_name', 'course_name',
          'specialization', 'pass_class', 'authority_name', 'issue_date']

TASK_START_TOKEN = '<s_cert>'
SPECIAL_TOKENS   = (
    [TASK_START_TOKEN, '</s_cert>'] +
    [f'<s_{f}>' for f in FIELDS] +
    [f'</s_{f}>' for f in FIELDS]
)

print(f'Loading Donut base model: {MODEL_NAME}')
processor = DonutProcessor.from_pretrained(MODEL_NAME)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Register task tokens
processor.tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set generation config to match task prompt
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids([TASK_START_TOKEN])[0]
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

# v5: align_long_axis=True handles real photos taken at different orientations
processor.image_processor.do_align_long_axis = True

# Memory optimisations for T4 free tier
model.config.use_cache = False
model.gradient_checkpointing_enable()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)

print(f'Model on {device}')
print(f'Vocab size: {len(processor.tokenizer):,}')
print(f'Decoder start token ID: {model.config.decoder_start_token_id}')
print(f'Special tokens: {SPECIAL_TOKENS}')

In [ ]:
# Section 3 (cont) — Dataset & DataLoader (v5)
import json, os, torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DATASET = '/content/dataset_v5'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

def build_target(row):
    """Build the XML-style target string for the decoder."""
    parts = ['<s_cert>']
    for f in FIELDS:
        v = row.get(f, '')
        parts.append(f'<s_{f}>{v}</s_{f}>')
    parts.append('</s_cert>')
    return ''.join(parts)


class CertDataset(Dataset):
    def __init__(self, split):
        with open(f'{DATASET}/metadata_{split}.jsonl') as fp:
            self.rows = [json.loads(l) for l in fp]
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row  = self.rows[idx]
        try:
            img = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
        except Exception:
            # Fallback to a blank image if the file is corrupt/missing
            img = Image.new('RGB', (1280, 960), (255, 255, 255))
        pv  = processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        tgt = build_target(row)
        lbl = processor.tokenizer(
            tgt, add_special_tokens=False,
            max_length=128, truncation=True, return_tensors='pt'
        ).input_ids.squeeze(0)
        return pv, lbl


def collate_fn(batch):
    pixels, labels_list = zip(*batch)
    pixels  = torch.stack(pixels)
    max_len = max(l.size(0) for l in labels_list)
    pad     = processor.tokenizer.pad_token_id
    padded  = torch.full((len(labels_list), max_len), pad, dtype=torch.long)
    for i, l in enumerate(labels_list):
        padded[i, :l.size(0)] = l
    padded[padded == pad] = -100
    return pixels, padded

BATCH    = 4
train_ds = CertDataset('train')
val_ds   = CertDataset('val')
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)

print(f'Dataset ready:')
print(f'  Train : {len(train_ds):,} | Val : {len(val_ds):,}')
print(f'  Batches/epoch: {len(train_dl):,}')

In [ ]:
# Section 3 (cont) — Training Loop (v5)
# 10 epochs | FP16 | Batch 4 x GradAccum 2 | Fixed checkpoint saving
import os, json, gc, torch, shutil
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

gc.collect()
torch.cuda.empty_cache()

NUM_EPOCHS    = 10
LR            = 3e-5
WARMUP_STEPS  = 200
GRAD_ACCUM    = 2
MAX_GRAD_NORM = 1.0
LOG_EVERY     = 50
DRIVE_CKPTS   = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v5'

# Ensure Drive folder exists BEFORE any write (avoids FUSE async race)
os.makedirs(DRIVE_CKPTS, exist_ok=True)
print(f'Checkpoint dir ready: {DRIVE_CKPTS}')

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * NUM_EPOCHS // GRAD_ACCUM
scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=5e-7)
scaler      = torch.amp.GradScaler('cuda')

step_global = 0
def get_lr():
    if step_global < WARMUP_STEPS:
        return LR * step_global / max(1, WARMUP_STEPS)
    return scheduler.get_last_lr()[0]


def _prune_optimizer_state(ckpt_root, current_epoch):
    # Keep ALL epoch model folders (never rmtree -- Drive FUSE is async).
    # Only remove optimizer.pt/scheduler.pt from older epochs to save space.
    import glob as _glob
    for d in sorted(_glob.glob(f'{ckpt_root}/epoch_*')):
        if d != f'{ckpt_root}/epoch_{current_epoch:02d}':
            for fname in ('optimizer.pt', 'scheduler.pt'):
                p = f'{d}/{fname}'
                if os.path.exists(p):
                    os.remove(p)
                    print(f'  [prune] {fname} removed from {d}')


def save_checkpoint(epoch, val_loss):
    ckpt_dir = f'{DRIVE_CKPTS}/epoch_{epoch:02d}'
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    torch.save(optimizer.state_dict(), f'{ckpt_dir}/optimizer.pt')
    torch.save(scheduler.state_dict(), f'{ckpt_dir}/scheduler.pt')
    with open(f'{ckpt_dir}/train_meta.json', 'w') as fp:
        json.dump({'epoch': epoch, 'val_loss': val_loss,
                   'step': step_global, 'lr': get_lr()}, fp, indent=2)
    print(f'  [ckpt] Saved epoch {epoch} -> {ckpt_dir} (val_loss={val_loss:.4f})')
    _prune_optimizer_state(DRIVE_CKPTS, epoch)


print(f'Training: {NUM_EPOCHS} epochs | {len(train_dl):,} batches/epoch')
print(f'  Batch {BATCH} x GradAccum {GRAD_ACCUM} = effective batch {BATCH*GRAD_ACCUM}')
print(f'  ~{len(train_dl) // 2} min/epoch | ~{NUM_EPOCHS * len(train_dl) // 2} min total\n')

history       = []
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\n=== Epoch {epoch}/{NUM_EPOCHS} ===')
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, (pixels, labels) in enumerate(train_dl):
        pixels = pixels.to(device)
        labels = labels.to(device)

        if step_global < WARMUP_STEPS:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * step_global / max(1, WARMUP_STEPS)

        with torch.amp.autocast('cuda'):
            loss = model(pixel_values=pixels, labels=labels).loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            if step_global >= WARMUP_STEPS:
                scheduler.step()
            optimizer.zero_grad()
            step_global += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            avg = train_loss / (batch_idx + 1)
            print(f'  step {step_global:5d} | batch {batch_idx+1}/{len(train_dl)} | loss={avg:.4f} | lr={get_lr():.2e}')

    avg_train = train_loss / len(train_dl)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for pixels, labels in val_dl:
            pixels, labels = pixels.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                val_loss += model(pixel_values=pixels, labels=labels).loss.item()
    val_loss /= len(val_dl)

    print(f'  Epoch {epoch}: train={avg_train:.4f} | val={val_loss:.4f}')
    history.append({'epoch': epoch, 'train_loss': avg_train, 'val_loss': val_loss})
    save_checkpoint(epoch, val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_dir = f'{DRIVE_CKPTS}/best'
        if os.path.exists(best_dir): shutil.rmtree(best_dir)
        shutil.copytree(f'{DRIVE_CKPTS}/epoch_{epoch:02d}', best_dir)
        for fname in ('optimizer.pt', 'scheduler.pt'):
            p = f'{best_dir}/{fname}'
            if os.path.exists(p): os.remove(p)
        print(f'  [best] New best val_loss={best_val_loss:.4f}')

print('\nTraining complete!')

## Section 4 — Resume Training (if Colab Disconnected)

Run this cell to resume from the **latest checkpoint** after any Colab timeout or restart.
This cell is fully self-contained — it mounts Drive, restores the dataset if needed,
and continues training where it left off.


In [ ]:
# Section 4 — Resume Training (v5, Self-Contained)
# Run this after any Colab timeout / disconnection / restart.
import os, sys, gc, json, glob, shutil, tempfile, torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

DRIVE_BASE  = '/content/drive/MyDrive/DegreeDetailExtract'
DRIVE_CKPTS = f'{DRIVE_BASE}/checkpoints_v5'
ZIP_PATH    = f'{DRIVE_BASE}/dataset_v5_archive.zip'
DATASET     = '/content/dataset_v5'
REPO        = '/content/DegreeDetailExtract'

# 1. Mount Drive if needed
if not os.path.exists('/content/drive/MyDrive'):
    print('Mounting Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(DRIVE_CKPTS):
    raise RuntimeError(
        f'Checkpoint directory not found: {DRIVE_CKPTS}\n'
        'Section 3 must have saved at least one epoch before Section 4 can resume.'
    )

# 2. Restore dataset if missing
meta_train        = f'{DATASET}/metadata_train.jsonl'
MIN_TRAIN_RECORDS = 4200

need_dataset = (
    not os.path.exists(meta_train)
    or sum(1 for _ in open(meta_train)) < MIN_TRAIN_RECORDS
)

if need_dataset:
    print(f'Dataset missing. Restoring from {ZIP_PATH} ...')
    if not os.path.exists(ZIP_PATH):
        raise RuntimeError(f'Archive not found: {ZIP_PATH}. Run Section 2.')
    import subprocess
    os.makedirs(DATASET, exist_ok=True)
    TMP = tempfile.mkdtemp(prefix='cert_extract_', dir='/content')
    try:
        subprocess.run(['unzip', '-qo', ZIP_PATH, '-d', TMP], check=True)
        for root, dirs, files in os.walk(TMP):
            dirs[:] = [d for d in dirs if not d.startswith('drive')]
            if 'metadata_train.jsonl' in files:
                for item in os.listdir(root):
                    dst = os.path.join(DATASET, item)
                    if os.path.exists(dst):
                        if os.path.isdir(dst): shutil.rmtree(dst)
                        else: os.remove(dst)
                    shutil.move(os.path.join(root, item), DATASET)
                break
        count = sum(1 for _ in open(meta_train)) if os.path.exists(meta_train) else 0
        print(f'Dataset restored: {count:,} train records.')
    finally:
        if os.path.exists(TMP): shutil.rmtree(TMP, ignore_errors=True)
else:
    print(f'Dataset ready: {sum(1 for _ in open(meta_train)):,} train records.')

# 3. Find latest checkpoint
epoch_dirs = sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*'))
if not epoch_dirs:
    raise RuntimeError(
        f'No epoch checkpoints at: {DRIVE_CKPTS}\n'
        'Re-run Section 3 from the beginning.'
    )

latest_dir = epoch_dirs[-1]
with open(f'{latest_dir}/train_meta.json') as fp:
    meta = json.load(fp)

completed_epochs = meta['epoch']
start_epoch      = completed_epochs + 1
step_global      = meta['step']
last_lr          = meta.get('lr', 1.2e-5)
best_val_loss    = meta.get('val_loss', float('inf'))
NUM_EPOCHS       = 10

print(f'\nLatest checkpoint : {latest_dir}')
print(f'  Completed : {completed_epochs} epochs')
print(f'  Resuming  : epoch {start_epoch}')
print(f'  Step      : {step_global}')

if start_epoch > NUM_EPOCHS:
    print(f'All {NUM_EPOCHS} epochs done. Go to Section 5.')
    raise SystemExit(0)

# 4. Load model
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'\nLoading model from {latest_dir} ...')
processor = DonutProcessor.from_pretrained(latest_dir)
model     = VisionEncoderDecoderModel.from_pretrained(latest_dir).to(device)
model.config.use_cache = False
model.gradient_checkpointing_enable()
print('Model loaded.')

# 5. Rebuild dataset
FIELDS = ['student_name', 'university_name', 'course_name',
          'specialization', 'pass_class', 'authority_name', 'issue_date']

def build_target(row):
    parts = ['<s_cert>']
    for f in FIELDS:
        parts.append(f'<s_{f}>{row.get(f, "")}</s_{f}>')
    parts.append('</s_cert>')
    return ''.join(parts)

class CertDataset(Dataset):
    def __init__(self, split):
        with open(f'{DATASET}/metadata_{split}.jsonl') as fp:
            self.rows = [json.loads(l) for l in fp]
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        try:
            img = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
        except Exception:
            img = Image.new('RGB', (1280, 960), (255, 255, 255))
        pv  = processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        lbl = processor.tokenizer(build_target(row), add_special_tokens=False,
                                   max_length=128, truncation=True,
                                   return_tensors='pt').input_ids.squeeze(0)
        return pv, lbl

def collate_fn(batch):
    pixels, labels_list = zip(*batch)
    pixels  = torch.stack(pixels)
    max_len = max(l.size(0) for l in labels_list)
    pad     = processor.tokenizer.pad_token_id
    padded  = torch.full((len(labels_list), max_len), pad, dtype=torch.long)
    for i, l in enumerate(labels_list):
        padded[i, :l.size(0)] = l
    padded[padded == pad] = -100
    return pixels, padded

BATCH    = 4
train_dl = DataLoader(CertDataset('train'), batch_size=BATCH, shuffle=True,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_dl   = DataLoader(CertDataset('val'), batch_size=BATCH, shuffle=False,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)
print(f'Data: {len(train_dl):,} train batches | {len(val_dl):,} val batches')

# 6. Optimizer & scheduler
GRAD_ACCUM    = 2
MAX_GRAD_NORM = 1.0
LOG_EVERY     = 50
remaining     = NUM_EPOCHS - completed_epochs
total_steps   = len(train_dl) * remaining // GRAD_ACCUM

optimizer = AdamW(model.parameters(), lr=last_lr, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, total_steps), eta_min=5e-7)

# Restore state if available
if os.path.exists(f'{latest_dir}/optimizer.pt'):
    optimizer.load_state_dict(torch.load(f'{latest_dir}/optimizer.pt', map_location=device))
    print('Optimizer state restored.')
if os.path.exists(f'{latest_dir}/scheduler.pt'):
    try:
        scheduler.load_state_dict(torch.load(f'{latest_dir}/scheduler.pt', map_location=device))
        print('Scheduler state restored.')
    except Exception as e:
        print(f'Scheduler restore failed ({e}), using fresh schedule.')

scaler = torch.amp.GradScaler('cuda')


def _prune_optimizer_state(ckpt_root, current_epoch):
    for d in sorted(glob.glob(f'{ckpt_root}/epoch_*')):
        if d != f'{ckpt_root}/epoch_{current_epoch:02d}':
            for fname in ('optimizer.pt', 'scheduler.pt'):
                p = f'{d}/{fname}'
                if os.path.exists(p): os.remove(p)


def save_checkpoint(epoch, val_loss):
    ckpt_dir = f'{DRIVE_CKPTS}/epoch_{epoch:02d}'
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    torch.save(optimizer.state_dict(), f'{ckpt_dir}/optimizer.pt')
    torch.save(scheduler.state_dict(), f'{ckpt_dir}/scheduler.pt')
    with open(f'{ckpt_dir}/train_meta.json', 'w') as fp:
        json.dump({'epoch': epoch, 'val_loss': val_loss,
                   'step': step_global, 'lr': scheduler.get_last_lr()[0]}, fp, indent=2)
    print(f'  [ckpt] Epoch {epoch} -> {ckpt_dir} (val={val_loss:.4f})')
    _prune_optimizer_state(DRIVE_CKPTS, epoch)


# 7. Training loop
print(f'\nResuming: Epoch {start_epoch} -> {NUM_EPOCHS}  (~{32 * remaining} min)\n')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f'\n=== Epoch {epoch}/{NUM_EPOCHS} (resumed) ===')
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, (pixels, labels) in enumerate(train_dl):
        pixels, labels = pixels.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            loss = model(pixel_values=pixels, labels=labels).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_global += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            avg = train_loss / (batch_idx + 1)
            print(f'  step {step_global:5d} | {batch_idx+1}/{len(train_dl)} | loss={avg:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

    avg_train = train_loss / len(train_dl)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for pixels, labels in val_dl:
            pixels, labels = pixels.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                val_loss += model(pixel_values=pixels, labels=labels).loss.item()
    val_loss /= len(val_dl)

    print(f'  Epoch {epoch}: train={avg_train:.4f} | val={val_loss:.4f}')
    save_checkpoint(epoch, val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_dir = f'{DRIVE_CKPTS}/best'
        if os.path.exists(best_dir): shutil.rmtree(best_dir)
        shutil.copytree(f'{DRIVE_CKPTS}/epoch_{epoch:02d}', best_dir)
        for fname in ('optimizer.pt', 'scheduler.pt'):
            p = f'{best_dir}/{fname}'
            if os.path.exists(p): os.remove(p)
        print(f'  [best] val={best_val_loss:.4f}')

print(f'\nDone! Epochs {start_epoch}-{NUM_EPOCHS} complete.')

## Section 5 — Evaluation (Synthetic Test Set)

In [ ]:
import os, glob, shutil, tempfile, torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/DegreeDetailExtract'
DRIVE_CKPTS = f'{DRIVE_BASE}/checkpoints_v5'
BEST        = f'{DRIVE_CKPTS}/best'
DATASET     = '/content/dataset_v5'
device      = 'cuda' if torch.cuda.is_available() else 'cpu'

if not os.path.exists(f'{DATASET}/metadata_test.jsonl'):
    import subprocess, tempfile
    ZIP = f'{DRIVE_BASE}/dataset_v5_archive.zip'
    if os.path.exists(ZIP):
        TMP = tempfile.mkdtemp(dir='/content')
        subprocess.run(['unzip', '-qo', ZIP, '-d', TMP], check=True)
        for root, dirs, files in __import__('os').walk(TMP):
            dirs[:] = [d for d in dirs if not d.startswith('drive')]
            if 'metadata_test.jsonl' in files:
                for item in __import__('os').listdir(root):
                    dst = f'{DATASET}/{item}'
                    if __import__('os.path', fromlist=['exists']).exists(dst):
                        if __import__('os.path', fromlist=['isdir']).isdir(dst): shutil.rmtree(dst)
                        else: __import__('os').remove(dst)
                    shutil.move(f'{root}/{item}', DATASET)
                break
        shutil.rmtree(TMP, ignore_errors=True)

if os.path.exists(BEST):
    load_dir = BEST
elif sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*')):
    load_dir = sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*'))[-1]
else:
    raise FileNotFoundError(f'No checkpoints at {DRIVE_CKPTS}')

print(f'Loading from: {load_dir}')
import warnings; warnings.filterwarnings('ignore')
import transformers; transformers.logging.set_verbosity_error()

processor = DonutProcessor.from_pretrained(load_dir)
model     = VisionEncoderDecoderModel.from_pretrained(load_dir).to(device)
if hasattr(model, 'generation_config') and model.generation_config:
    model.generation_config.max_length = None
model.eval()
print(f'Model loaded on {device}.')

In [ ]:
import json, re, editdistance
from PIL import Image
import torch, warnings, transformers
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

DATASET = '/content/dataset_v5'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

with open(f'{DATASET}/metadata_test.jsonl') as fp:
    test_rows = [json.loads(l) for l in fp]

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)

exact  = {f: 0 for f in FIELDS}
cer    = {f: [] for f in FIELDS}

for idx, row in enumerate(test_rows):
    if (idx + 1) % 25 == 0:
        print(f'  Evaluating {idx+1}/{len(test_rows)}...', end='\r')
    img    = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)
    dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                                   return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                             max_new_tokens=128, early_stopping=True)
    pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

    for f in FIELDS:
        gt   = str(row.get(f, '')).lower().strip()
        m    = re.search(fr'<s_{f}>(.*?)</s_{f}>', pred_str, re.DOTALL)
        pred = m.group(1).strip().lower() if m else ''
        if pred == gt: exact[f] += 1
        if gt: cer[f].append(editdistance.eval(pred, gt) / max(len(gt), 1))

n = len(test_rows)
print(f'\nEvaluation on {n} SYNTHETIC test certificates')
print('NOTE: this only measures in-distribution accuracy.')
print(f'\n  Field{" ":18s} Exact %  Avg CER')
print('  ' + '-' * 40)
for f in FIELDS:
    ex_pct  = 100 * exact[f] / n
    avg_cer = 100 * (sum(cer[f]) / len(cer[f])) if cer[f] else 0
    bar     = '|' * int(ex_pct // 6)
    print(f'  {f:<24} {ex_pct:5.1f}%  {avg_cer:5.1f}%  {bar}')

In [ ]:
# Show 3 test examples side-by-side: predicted vs ground truth
import random, re, json, warnings, transformers
from PIL import Image
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

DATASET = '/content/dataset_v5'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

with open(f'{DATASET}/metadata_test.jsonl') as fp:
    test_rows = [json.loads(l) for l in fp]

samples = random.sample(test_rows, 3)
fig, axes = plt.subplots(1, 3, figsize=(18, 9))

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)

for ax, row in zip(axes, samples):
    img    = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)
    dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                                   return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                             max_new_tokens=128, early_stopping=True)
    pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

    annotation = ''
    for f in FIELDS:
        gt = str(row.get(f, ''))
        m  = re.search(fr'<s_{f}>(.*?)</s_{f}>', pred_str, re.DOTALL)
        pred = m.group(1).strip() if m else 'MISSING'
        icon = 'OK' if pred.lower() == gt.lower() else 'X'
        annotation += f'[{icon}] {f[:14]}: {pred[:22]}\n'

    ax.imshow(img)
    ax.set_title(annotation, fontsize=6.5, family='monospace', loc='left')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Section 6 — Inference on a Real Certificate

In [ ]:
from google.colab import files
uploaded = files.upload()
real_cert_path = list(uploaded.keys())[0]
print(f'Uploaded: {real_cert_path}')
from PIL import Image
img = Image.open(real_cert_path)
img.show()
print(f'Size: {img.size}')

In [ ]:
import os, glob, shutil, torch, re, json, warnings, transformers
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_CKPTS  = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v5'
BEST         = f'{DRIVE_CKPTS}/best'
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.exists(BEST):
    load_dir = BEST
elif sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*')):
    load_dir = sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*'))[-1]
else:
    raise FileNotFoundError(f'No checkpoints at {DRIVE_CKPTS}')

print(f'Loading from: {load_dir}')
processor = DonutProcessor.from_pretrained(load_dir)
model     = VisionEncoderDecoderModel.from_pretrained(load_dir).to(device)
if hasattr(model, 'generation_config') and model.generation_config:
    model.generation_config.max_length = None
model.eval()

FIELDS = ['student_name', 'university_name', 'course_name',
          'specialization', 'pass_class', 'authority_name', 'issue_date']

img    = Image.open(real_cert_path).convert('RGB')
pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)
dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                               return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                         max_new_tokens=128, early_stopping=True)

pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)
result   = {}
for f in FIELDS:
    m = re.search(fr'<s_{f}>(.*?)</s_{f}>', pred_str, re.DOTALL)
    result[f] = m.group(1).strip() if m else ''

print(json.dumps(result, indent=2, ensure_ascii=False))
missing = [f for f in FIELDS if not result.get(f)]
if missing: print('\nMissing fields:', missing)
else: print('\nAll 7 fields extracted.')

## Section 7 — Real-World Evaluation

Runs the SAME field-exact-match / CER metric as Section 5, but on the
**real certificates from `real_certs/`** that were held out from training
(placed in the test split). This is the number to report and trust.

> The gap between Section 5 (synthetic) and Section 7 (real) metrics is expected
> and worth discussing in your report — it quantifies the remaining domain gap.


In [ ]:
import json, re, editdistance, os, glob, torch, warnings, transformers
from PIL import Image

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

DATASET = '/content/dataset_v5'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

# Load ONLY the real cert rows from test split
with open(f'{DATASET}/metadata_test.jsonl') as fp:
    all_test = [json.loads(l) for l in fp]

real_test = [r for r in all_test if r.get('is_real')]
print(f'Real-world test set: {len(real_test)} certificates')

if not real_test:
    print('No real certs in test split. Try the full set:')
    real_test = all_test  # fallback to everything

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)

exact = {f: 0 for f in FIELDS}
cer   = {f: [] for f in FIELDS}

for idx, row in enumerate(real_test):
    img    = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)
    dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                                   return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                             max_new_tokens=128, early_stopping=True)
    pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

    for f in FIELDS:
        gt   = str(row.get(f, '')).lower().strip()
        m    = re.search(fr'<s_{f}>(.*?)</s_{f}>', pred_str, re.DOTALL)
        pred = m.group(1).strip().lower() if m else ''
        if pred == gt: exact[f] += 1
        if gt: cer[f].append(editdistance.eval(pred, gt) / max(len(gt), 1))

n = len(real_test)
print(f'\nREAL-WORLD evaluation on {n} held-out real certificates')
print(f'\n  Field{" ":18s} Exact %  Avg CER')
print('  ' + '-' * 44)
for f in FIELDS:
    ex_pct  = 100 * exact[f] / n if n else 0
    avg_cer = 100 * (sum(cer[f]) / len(cer[f])) if cer[f] else 0
    bar     = '|' * int(ex_pct // 6)
    print(f'  {f:<24} {ex_pct:5.1f}%  {avg_cer:5.1f}%  {bar}')

print('\nCompare this table to Section 5.')
print('A large gap is expected and worth discussing in your report.')